# 09 Evaluasi lintas domain: CodeQA Java, ClassEvalQA, CS1QA
Model CodeT5-small Python (seleksi BERTScore) diterapkan tanpa training ulang pada set lain, kondisi FULL, QONLY, dan kode diacak.
Urutan: konversi data eksternal ke format CodeQA, decoding, BERTScore, lalu cek tumpang tindih Java-Python.

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## ext_convert.py
Konversi ClassEvalQA dan CS1QA (datapaf) ke berkas .code/.question/.answer di ext_data/.

In [ ]:
ARGS = []
sys.argv = ['ext_convert.py']

**Sumber:** code/ext_convert.py

```text
Convert ClassEvalQA and CS1QA (datapaf unified set) into CodeQA-style line files.

Code is normalised roughly like CodeQA Python: comments and docstrings removed, punctuation dropped,
identifiers split on _ / camelCase / digits, string literals kept. shortcut: regex approximation of the
CodeQA preprocessing, not the original pipeline.
Writes ext_data/<name>/test.{code,question,answer} and prints overlap with CodeQA Python train.
```

In [ ]:
import json, re
from pathlib import Path
import pandas as pd

In [ ]:
ROOT = Path.home() / "inahuy/kacuy"
RAW = ROOT / "ext_data/raw"
OUT = ROOT / "ext_data"
TRAIN = ROOT / "kc-codeqa-reproduce/data/python/train"

TRIPLE = re.compile(r'("""|\'\'\')[\s\S]*?\1')
COMMENT = re.compile(r"#[^\n]*")
TOKEN = re.compile(r"'[^'\n]*'|\"[^\"\n]*\"|[A-Za-z_][A-Za-z0-9_]*|\d+(?:\.\d+)?")
SUB = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+")

In [ ]:
def norm_code(src):
    src = COMMENT.sub(" ", TRIPLE.sub(" ", src))
    out = []
    for t in TOKEN.findall(src):
        if t[0] in "'\"":
            out.append(" ".join(t.split()))
        else:
            out.extend(SUB.findall(t) or [t])
    return " ".join(out)

In [ ]:
one_line = lambda s: " ".join(str(s).split())

assert norm_code("def str2css(sourceString, x_y=1):  # c\n    '''doc'''\n    return StringIO('a b')") == \
    "def str 2 css source String x y 1 return String IO 'a b'"

In [ ]:
def write(name, rows):
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    rows = [r for r in rows if norm_code(r["code"]) and one_line(r["question"]) and one_line(r["answer"])]
    for ext, f in (("code", lambda r: norm_code(r["code"])), ("question", lambda r: one_line(r["question"])),
                   ("answer", lambda r: one_line(r["answer"]))):
        (d / f"test.{ext}").write_text("\n".join(f(r) for r in rows) + "\n", encoding="utf-8")
    tq = (TRAIN / "train.question").read_text(encoding="utf-8").splitlines()
    ta = (TRAIN / "train.answer").read_text(encoding="utf-8").splitlines()
    tc = {" ".join(c.split()) for c in (TRAIN / "train.code").read_text(encoding="utf-8").splitlines()}
    qs, qas = {q.strip() for q in tq}, {(q.strip(), a.strip()) for q, a in zip(tq, ta)}
    q = [one_line(r["question"]) for r in rows]
    a = [one_line(r["answer"]) for r in rows]
    c = [norm_code(r["code"]) for r in rows]
    n = len(rows)
    print(json.dumps({"name": name, "n": n,
                      "question_seen_in_codeqa_train_pct": round(100 * sum(x in qs for x in q) / n, 2),
                      "qa_pair_seen_pct": round(100 * sum((x, y) in qas for x, y in zip(q, a)) / n, 2),
                      "code_seen_pct": round(100 * sum(x in tc for x in c) / n, 2),
                      "answer_tokens_mean": round(sum(len(x.split()) for x in a) / n, 2),
                      "code_tokens_median": sorted(len(x.split()) for x in c)[n // 2],
                      "example": {"q": q[0], "a": a[0], "code": c[0][:200]}}))

In [ ]:
p = pd.read_parquet(RAW / "ClassEvalQABenchmark/test.parquet")
write("classevalqa", [dict(code=r.method_code, question=r.question, answer=r.answer) for r in p.itertuples()])

cs = []
for s in ("train", "dev", "test"):
    cs += [r for r in json.loads((RAW / f"CodeQuestionAnswering/{s}_filtered2.json").read_text()) if r["src"] == "cs1qa"]
write("cs1qa", cs)

## xdomain_codet5.py

In [ ]:
VARIANTS = {
    'java_full': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java/test', '--run-tag', 'xdomain_java_full_bertsel'],
    'java_qonly': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_qonly_bertsel', '--mode', 'qonly', '--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java/test', '--run-tag', 'xdomain_java_qonly_bertsel'],
    'java_full_shuffled': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java/test', '--run-tag', 'xdomain_java_full_bertsel_shuffled', '--shuffle'],
    'classevalqa_full': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/classevalqa', '--run-tag', 'xdomain_classevalqa_full_bertsel'],
    'classevalqa_qonly': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_qonly_bertsel', '--mode', 'qonly', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/classevalqa', '--run-tag', 'xdomain_classevalqa_qonly_bertsel'],
    'classevalqa_full_shuffled': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/classevalqa', '--run-tag', 'xdomain_classevalqa_full_bertsel_shuffled', '--shuffle'],
    'cs1qa_full': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/cs1qa', '--run-tag', 'xdomain_cs1qa_full_bertsel'],
    'cs1qa_qonly': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_qonly_bertsel', '--mode', 'qonly', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/cs1qa', '--run-tag', 'xdomain_cs1qa_qonly_bertsel'],
    'cs1qa_full_shuffled': ['--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--mode', 'full', '--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/cs1qa', '--run-tag', 'xdomain_cs1qa_full_bertsel_shuffled', '--shuffle'],
}
RUN = 'classevalqa_full'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['xdomain_codet5.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/xdomain_codet5.py

In [ ]:
import argparse, json
from pathlib import Path
import torch
import kc_common as kc
from transformers import AutoTokenizer, T5ForConditionalGeneration

In [ ]:
p = argparse.ArgumentParser()
p.add_argument("--run-dir", required=True)
p.add_argument("--mode", choices=("full", "qonly"), required=True)
p.add_argument("--data", required=True, help="folder with <split>.code/.question/.answer")
p.add_argument("--split", default="test")
p.add_argument("--run-tag", required=True)
p.add_argument("--ckpt", default="best_bertscore")
p.add_argument("--batch", type=int, default=16)
p.add_argument("--shuffle", action="store_true", help="derange the code column within the split, seed 42 (E2 rule)")
args = p.parse_args(ARGS)

run_dir = Path(args.run_dir)
cfg = json.loads((run_dir / "config.json").read_text())
data = Path(args.data)

In [ ]:
def read(ext):
    return (data / f"{args.split}.{ext}").read_text(encoding="utf-8").splitlines()

In [ ]:
C, Q, A = read("code"), read("question"), read("answer")
assert len(C) == len(Q) == len(A), (len(C), len(Q), len(A))
rows = [{"id": i, "code": c.strip(), "question": q.strip(), "answer": a.strip()} for i, (c, q, a) in enumerate(zip(C, Q, A))]

if args.shuffle:
    import random
    codes = [r["code"] for r in rows]
    n = len(codes)
    rng = random.Random(42)
    perm = list(range(n))
    for _ in range(1000):
        rng.shuffle(perm)
        if all(perm[i] != i and codes[perm[i]] != codes[i] for i in range(n)):
            break
    else:
        raise RuntimeError("no derangement")
    for i, r in enumerate(rows):
        r["code"] = codes[perm[i]]
    kc.write_json(kc.EXP / "maps" / f"{args.run_tag}_shuffle_seed42.json", {"donor_id": perm, "n": n})
tok = AutoTokenizer.from_pretrained(run_dir / args.ckpt)
model = T5ForConditionalGeneration.from_pretrained(run_dir / args.ckpt).to("cuda").eval()
srcs = [kc.codet5_source(r["question"], r["code"], args.mode) for r in rows]
enc = tok(srcs, max_length=cfg["max_source"], truncation=True)["input_ids"]
order = sorted(range(len(enc)), key=lambda i: -len(enc[i]))
preds = [None] * len(enc)

In [ ]:
for s in range(0, len(order), args.batch):
    idx = order[s:s + args.batch]
    width = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), width), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for j, i in enumerate(idx):
        ids[j, :len(enc[i])] = torch.tensor(enc[i]); mask[j, :len(enc[i])] = 1
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        gen = model.generate(input_ids=ids.cuda(), attention_mask=mask.cuda(), num_beams=10,
                             max_new_tokens=cfg["generation_max_new_tokens"], do_sample=False)
    for i, t in zip(idx, tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)):
        preds[i] = t.strip()

metrics, per = kc.evaluate(preds, [r["answer"] for r in rows])
out = kc.EXP / "runs" / args.run_tag
out.mkdir(parents=True, exist_ok=True)
config = {"experiment": "XDOMAIN", "model": "codet5", "mode": args.mode, "data": str(data), "split": args.split, "source_run": str(run_dir), "ckpt": args.ckpt, "decoding": "beam10", "max_source": cfg["max_source"], "n": len(rows)}
kc.write_json(out / "results.json", {"run_tag": args.run_tag, **config, args.split: metrics})
kc.write_predictions(out / f"{args.split}_predictions.jsonl", rows, preds, per, args.mode.upper(), [r["id"] for r in rows])
run = kc.wandb_init(out, args.run_tag, "XDOMAIN", ["codet5", args.mode, "java", "seed42"], config, "inference")
records = kc.read_jsonl(out / f"{args.split}_predictions.jsonl")
kc.wandb_finalize(run, out, args.run_tag, metrics, records)
run.finish()
print("DONE", args.run_tag, json.dumps({k: round(metrics[k], 3) for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}), flush=True)

### Ringkasan hasil test (dibaca dari runs/<tag>/results.json)

In [ ]:
TAGS = ['xdomain_java_full_bertsel', 'xdomain_java_qonly_bertsel', 'xdomain_java_full_bertsel_shuffled', 'xdomain_classevalqa_full_bertsel', 'xdomain_classevalqa_qonly_bertsel', 'xdomain_classevalqa_full_bertsel_shuffled', 'xdomain_cs1qa_full_bertsel', 'xdomain_cs1qa_qonly_bertsel', 'xdomain_cs1qa_full_bertsel_shuffled']
for tag in TAGS:
    for cand in (tag, "smoke_" + tag):
        f = EXP / "runs" / cand / "results.json"
        if f.exists():
            r = json.loads(f.read_text())
            t = r.get("test", {})
            print(cand, {k: round(v, 2) for k, v in t.items() if isinstance(v, (int, float))})
            break
    else:
        print(tag, "-> belum ada results.json")

## bertscore_xdomain.py
Varian 'ext' sama dengan run_ext.sh. Varian 'java' menulis ke berkas baru agar tidak menimpa hasil lama.

In [ ]:
VARIANTS = {
    'ext': ['--set', 'classevalqa_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_classevalqa_full_bertsel/test_predictions.jsonl', '--set', 'classevalqa_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_classevalqa_qonly_bertsel/test_predictions.jsonl', '--set', 'classevalqa_full_bertsel_shuffled=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_classevalqa_full_bertsel_shuffled/test_predictions.jsonl', '--set', 'cs1qa_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_cs1qa_full_bertsel/test_predictions.jsonl', '--set', 'cs1qa_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_cs1qa_qonly_bertsel/test_predictions.jsonl', '--set', 'cs1qa_full_bertsel_shuffled=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_cs1qa_full_bertsel_shuffled/test_predictions.jsonl', '--out', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/bertscore_ext.json'],
    'java': ['--set', 'java_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_java_full_bertsel/test_predictions.jsonl', '--set', 'java_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_java_qonly_bertsel/test_predictions.jsonl', '--set', 'java_full_bertsel_shuffled=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/xdomain_java_full_bertsel_shuffled/test_predictions.jsonl', '--out', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/bertscore_xdomain_java.json'],
}
RUN = 'ext'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['bertscore_xdomain.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/bertscore_xdomain.py

In [ ]:
import argparse, json
from pathlib import Path
import bert_score

In [ ]:
E = "/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp"
p = argparse.ArgumentParser()
p.add_argument("--set", action="append", required=True, help="name=path to test_predictions.jsonl")
p.add_argument("--out", required=True)
args = p.parse_args(ARGS)
out = {}
for item in args.set:
    name, path = item.split("=", 1)
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    P, R, F = bert_score.score([r["prediction"] for r in rows], [r["reference"] for r in rows],
                               lang="en", batch_size=128, verbose=False, device="cuda")
    out[name] = {"path": path, "n": len(rows), "BERTScore_P": 100 * P.mean().item(), "BERTScore_R": 100 * R.mean().item(), "BERTScore_F1": 100 * F.mean().item()}
    print("DONE", name, json.dumps(out[name]), flush=True)
Path(args.out).write_text(json.dumps(out, indent=2), encoding="utf-8")
print("WROTE", args.out, flush=True)

## xl_overlap.py
Tumpang tindih string antara data Java dan Python CodeQA.

In [ ]:
ARGS = []
sys.argv = ['xl_overlap.py']

**Sumber:** code/xl_overlap.py

In [ ]:
from pathlib import Path
from collections import Counter

In [ ]:
J=Path('/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java')
P=Path('/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/python')

In [ ]:
def rd(d,s,e): return [x.strip() for x in (d/s/(s+'.'+e)).read_text(encoding='utf-8').splitlines()]

In [ ]:
jq=rd(J,'test','question'); ja=rd(J,'test','answer')
trq=rd(P,'train','question'); tra=rd(P,'train','answer')
pq=set(trq); pa=set(tra); pqa=set(zip(trq,tra))
print('JQ_IN_PYTRAIN_Q', sum(q in pq for q in jq), len(jq))
print('JA_IN_PYTRAIN_A', sum(a in pa for a in ja), len(ja))
print('JQA_IN_PYTRAIN_QA', sum((q,a) in pqa for q,a in zip(jq,ja)))
print('TOPANS', Counter(ja).most_common(4))